In [1]:
import numpy as np, networkx as nx, os, random, pandas as pd
import test_WTM as wtm
import gudhi_persistence as gp
import utilsA1 as utils
import importlib

importlib.reload(utils)
importlib.reload(wtm)
importlib.reload(gp)


PATH = os.getcwd()
output_file = "contagion_model_graphics"
print(PATH)

U:\Academic\NetworkModels\Project1\A1


In [57]:
importlib.reload(utils)
importlib.reload(wtm)
importlib.reload(gp)

params_list = {'num_nodes': 100, 'num_neighbor_nodes': 1,
                    'total_random_edges': 4, 'distance_threshold': 2, 'weighted': False,
                    'ngeo_placement': 'ngeo_per_node', 'n_seeds': 1, 'node_active_threshold': 0.1,
                    'upper_weight_limit': 20, 'skew_power': 2, 'seed_cluster_distance': 10,
                    'seeding_method': 'all_combinations',
                    'ngeom_edges_in_persistence': False, 'max_persistence_dim': 2}

params_list['threshold_sum']= sum(range(params_list['num_nodes'])) - 400
np.random.seed(666)
random.seed(666)


df, activation_results = wtm.main_sims(params_list = [params_list],
                   output_file=output_file, save_files=False)

In [58]:
df

,simulation_id,realization_id,num_nodes,time,state,state_abnormal_sum,num_active_nodes,active_nodes,node_active_threshold,H0,...,total_geo_edges,total_non_geo_edges,num_seeds,seed_nodes,seed_cluster_distance,weighted,average_weight_per_edge,skew_power,upper_weight_limit,distance_threshold
0,0,0,100,0,0,4550,1,"[0.0, nan, nan, nan, nan, nan, nan, nan, nan, ...",0.1,1,...,100,200,1,"(0,)",10,False,0.0,2,20,2
1,0,0,100,1,0,4550,7,"[0.0, 1.0, nan, nan, nan, nan, nan, nan, nan, ...",0.1,5,...,100,200,1,"(0,)",10,False,0.0,2,20,2
2,0,0,100,2,0,4550,33,"[0.0, 1.0, 2.0, 2.0, nan, nan, 2.0, nan, nan, ...",0.1,17,...,100,200,1,"(0,)",10,False,0.0,2,20,2
3,0,0,100,3,0,4550,90,"[0.0, 1.0, 2.0, 2.0, 3.0, 3.0, 2.0, 3.0, 3.0, ...",0.1,9,...,100,200,1,"(0,)",10,False,0.0,2,20,2
4,0,0,100,4,1,4550,100,"[0.0, 1.0, 2.0, 2.0, 3.0, 3.0, 2.0, 3.0, 3.0, ...",0.1,1,...,100,200,1,"(0,)",10,False,0.0,2,20,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,0,99,100,0,0,4550,1,"[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...",0.1,1,...,100,200,1,"(99,)",10,False,0.0,2,20,2
496,0,99,100,1,0,4550,7,"[1.0, nan, nan, 1.0, nan, nan, 1.0, nan, nan, ...",0.1,5,...,100,200,1,"(99,)",10,False,0.0,2,20,2
497,0,99,100,2,0,4550,31,"[1.0, 2.0, 2.0, 1.0, 2.0, 2.0, 1.0, 2.0, nan, ...",0.1,14,...,100,200,1,"(99,)",10,False,0.0,2,20,2
498,0,99,100,3,0,4550,83,"[1.0, 2.0, 2.0, 1.0, 2.0, 2.0, 1.0, 2.0, 3.0, ...",0.1,12,...,100,200,1,"(99,)",10,False,0.0,2,20,2


In [45]:
grouped = df.groupby(['simulation_id', 'realization_id'])

records = []
for sim_id, group in grouped:
    abnormal_records = group[group['state'] == 1]
    max_h0 = group['H0'].max()
    max_h1 = group['H1'].max()
    if not abnormal_records.empty:
        row = abnormal_records.iloc[0]
        time_to_event = row['time']
        state = 1
    else:
        row = group.iloc[-1]
        time_to_event = row['time']
        state = 0

    features = row.drop(['time',  'num_nodes']).to_dict()

    records.append({
        'duration': time_to_event,
        'state': state,
        'max_h0': max_h0,
        'max_h1': max_h1,
        **features
    })

cox_df = pd.DataFrame(records)

In [47]:
cox_df

,duration,state,max_h0,max_h1,simulation_id,realization_id,state_abnormal_sum,num_active_nodes,active_nodes,node_active_threshold,...,H2,num_non_geo_edges,num_seeds,seed_nodes,seed_cluster_distance,weighted,average_weight_per_edge,skew_power,upper_weight_limit,distance_threshold
0,35,1,3,1,0,0,6740,117,"[0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, ...",0.1,...,0,2,1,"(0,)",10,False,0.0,2,20,2
1,34,1,3,1,0,1,6740,115,"[1.0, 0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, ...",0.1,...,0,2,1,"(1,)",10,False,0.0,2,20,2
2,34,1,3,1,0,2,6740,117,"[2.0, 1.0, 0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, ...",0.1,...,0,2,1,"(2,)",10,False,0.0,2,20,2
3,34,1,3,1,0,3,6740,118,"[3.0, 2.0, 1.0, 0.0, 1.0, 2.0, 3.0, 4.0, 5.0, ...",0.1,...,0,2,1,"(3,)",10,False,0.0,2,20,2
4,33,1,3,1,0,4,6740,116,"[4.0, 3.0, 2.0, 1.0, 0.0, 1.0, 2.0, 3.0, 4.0, ...",0.1,...,0,2,1,"(4,)",10,False,0.0,2,20,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,39,1,3,1,0,115,6740,115,"[5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13...",0.1,...,0,2,1,"(115,)",10,False,0.0,2,20,2
116,38,1,3,1,0,116,6740,115,"[4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12....",0.1,...,0,2,1,"(116,)",10,False,0.0,2,20,2
117,37,1,3,1,0,117,6740,115,"[3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0...",0.1,...,0,2,1,"(117,)",10,False,0.0,2,20,2
118,36,1,3,1,0,118,6740,115,"[2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0,...",0.1,...,0,2,1,"(118,)",10,False,0.0,2,20,2


In [ ]:
## Contagion Single Realization Test

## Contagion Single Realization Test

In [28]:
importlib.reload(wtm)
importlib.reload(utils)
importlib.reload(gp)


params_temp_list = {'num_nodes': 50, 'num_neighbor_nodes': 1,
                    'total_random_edges': 2, 'distance_threshold': 2, 'weighted': False,
                    'ngeo_placement': 'ngeo_per_node', 'n_seeds': 2, 'node_active_threshold': 0.01,
                    'upper_weight_limit': 5, 'skew_power': 3, 'seed_cluster_distance': 10,
                    'ngeom_edges_in_persistence': False, 'max_persistence_dim': 2,
                    'seeding_method': 'cluster_seeding'}

params_temp_list['threshold_sum']= sum(range(params_temp_list['num_nodes'])) - 1

G, seed_nodes = wtm.simulate_contagion_map(params=params_temp_list)
graph, snapshots, activation_times, results = wtm.simulate_contagion_realization(graph = G, init_seeds = seed_nodes, params = params_temp_list,
max_steps = 100, sim_id = 1, realization_id = 1)

betti_numbers, simplex_intervals, direct_intervals_gudhi = gp.compute_persistence(graph=graph, activation_times=activation_times, max_dim=2)

In [31]:
direct_intervals_gudhi

array([[((0.0, 10), (0.0, 10)), (), ()],
       [((1.0, 10), (1.0, 10), (1.0, 10), (1.0, 10), (1.0, 10)), (), ()],
       [((1.0, 10), (2.0, 10), (2.0, 10), (2.0, 10), (2.0, 10), (2.0, 10), (2.0, 10), (2.0, 10), (1.0, 10)),
        (), ()],
       [((2.0, 3.0), (2.0, 10), (1.0, 10), (1.0, 10)), (), ()],
       [((2.0, 3.0), (2.0, 4.0), (1.0, 4.0), (1.0, 10)), ((4.0, 10),),
        ()]], dtype=object)

In [27]:
max_timestep = (direct_intervals_gudhi.shape[0] - 1) + 1
max_timestep

5

In [10]:
simplex_intervals

defaultdict(list,
            {0: [(0, [(0.0, inf), (0.0, inf)]),
              (1,
               [(1.0, inf),
                (1.0, inf),
                (1.0, inf),
                (1.0, inf),
                (1.0, inf),
                (1.0, inf)]),
              (2,
               [(1.0, inf), (2.0, inf), (2.0, inf), (2.0, inf), (1.0, inf)]),
              (3,
               [(2.0, 3.0), (2.0, 3.0), (1.0, inf), (3.0, inf), (1.0, inf)]),
              (4,
               [(2.0, 3.0), (2.0, 3.0), (1.0, 4.0), (3.0, 4.0), (1.0, inf)])],
             1: [(0, []), (1, []), (2, []), (3, []), (4, [(4.0, inf)])],
             2: [(0, []), (1, []), (2, []), (3, []), (4, [])]})

In [16]:
from gudhi.representations import Landscape

D = direct_intervals_gudhi
diag=  [D]
l = Landscape(num_landscapes=2, resolution=10).fit_transform(diag)
print(l)

IndexError: too many indices for array: array is 1-dimensional, but 2 were indexed